**This notebook shows another example of how data was labeled with the keyword classifier**

In [1]:
import pandas as pd
import multiprocessing as mp
import numpy as np
import regex as re
import swifter
import html

df = pd.read_parquet('conservative_commentsClean.parquet')
df = df[~df['body'].isin(['removed', 'deleted'])]

/Users/emma/anaconda3/envs/macs30123/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
labeled_data = pd.read_csv("~/Desktop/labeled_data.csv")
labeled_data2 = pd.read_csv("~/Desktop/pl_review22.csv")

labeled_data = pd.concat([labeled_data, labeled_data2], ignore_index=True)

In [85]:
reviewed = pd.read_csv('~/Desktop/review.csv')


In [86]:
reviewed

,body,author,score,id,link_id,subreddit,author_flair_text,time,keyword_label,my_label
0,its incredibly long considering the english mo...,DuvalHeart,6,hf9vveg,t3_q0fu96,politics,florida,2021-10-03 21:15:47,no stance,NaN
1,why we exclusively vote for prolife candidates...,RipplePark,8,iuyfrgz,t3_ylhg7y,politics,NaN,2022-11-03 23:05:38,no stance,Unclear
2,wheres that guy that works a story about shitt...,we_have_met_before,2,dqoys13,t3_7h81dt,politics,NaN,2017-12-03 06:58:08,no stance,NaN
3,other than the extreme i find most religious p...,pzerr,1,ixeiepa,t3_z1rs2z,politics,NaN,2022-11-22 20:57:53,no stance,prochoice
4,thats not true at all. dems are just as guilty...,SeaExisting2304,0,idobrbl,t3_vjt5ca,politics,NaN,2022-06-25 11:12:23,no stance,prochoice
...,...,...,...,...,...,...,...,...,...,...
145,yeah this is an example of someone lying to yo...,LegacyAngel,0,ietl7j3,t3_vqttp7,politics,NaN,2022-07-04 14:28:49,no stance,NaN
146,and polling makes people feel good.,gnomebludgeon,1,i7a9cc4,t3_ui4sqr,politics,NaN,2022-05-04 12:36:15,no stance,NaN
147,or which plants to smoke. or what flavors you ...,ifandbut,4,h0fali5,t3_nr25ff,politics,NaN,2021-06-03 10:52:37,no stance,NaN
148,dont need a passport to enter canada as a us c...,Boccaperta,2,i793olv,t3_uhudjb,politics,NaN,2022-05-04 04:12:32,no stance,NaN


In [87]:
labeled_data = pd.concat([labeled_data, reviewed], ignore_index=True)


In [88]:
labeled_data

,body,author,score,id,parent_id,link_id,subreddit,author_flair_text,time,keyword_label,my_label
0,gods judgement on nations is different than pe...,ConHawthorne,12,g5qg79t,t1_g5qd1jo,t3_ivbo4e,Conservative,catholic conservative,2020-09-18 19:39:29,prolife,prolife
1,no i dont. but i sometimes feel like a burden ...,Fantasie-Sign,1,dvlo3mi,t1_dvlnza7,t3_83qbg5,Conservative,conservative,2018-03-12 23:19:36,prolife,prolife
2,im pro human in general. i pretty much stop b...,RedBaronsBrother,0,ffj8uxe,t1_ffj6ji8,t3_etqwfq,Conservative,conservative,2020-01-25 22:59:57,prolife,prolife
3,iuds can specifically prevent implantation. o...,CarsomyrPlusSix,0,gyu45ee,t1_gyq167a,t3_nga3yu,Conservative,paleoconservative libertarian,2021-05-20 15:59:53,prolife,prolife
4,the little worm snivels about dehumanization a...,CarsomyrPlusSix,2,h6dh7zj,t1_h6deyli,t3_oqhpaq,Conservative,paleoconservative libertarian,2021-07-24 15:31:50,prolife,prolife
...,...,...,...,...,...,...,...,...,...,...,...
795,yeah this is an example of someone lying to yo...,LegacyAngel,0,ietl7j3,NaN,t3_vqttp7,politics,NaN,2022-07-04 14:28:49,no stance,NaN
796,and polling makes people feel good.,gnomebludgeon,1,i7a9cc4,NaN,t3_ui4sqr,politics,NaN,2022-05-04 12:36:15,no stance,NaN
797,or which plants to smoke. or what flavors you ...,ifandbut,4,h0fali5,NaN,t3_nr25ff,politics,NaN,2021-06-03 10:52:37,no stance,NaN
798,dont need a passport to enter canada as a us c...,Boccaperta,2,i793olv,NaN,t3_uhudjb,politics,NaN,2022-05-04 04:12:32,no stance,NaN


In [89]:
labeled_data['my_label'] = labeled_data['my_label'].str.lower()

In [84]:
labeled_data.to_csv('labeled_data.csv', index=False)

In [19]:

def remove_urls(text):
    text = re.sub(r"http[s]?://\S+", "link", text)  # Remove http or https links
    text = re.sub(r"www\.\S+", "link", text)        # Remove www. links
    text = re.sub(r"\S+\.com\S*", "link", text)     # Remove .com domains
    return text.strip()

labeled_data['body'] = labeled_data['body'].apply(remove_urls)
df['body'] = df['body'].apply(remove_urls)

In [20]:
df_unlabeled = df[~df['id'].isin(labeled_data['id'])]

# running again

In [ ]:
pck = [
    "my choice", "bodily autonomy", "body autonomy", "womens rights", "reproductive rights",
    "abortion rights", "forced birth", "government control", "privacy", "unsafe abortions", 'christofacists', 
    "anti choice", "im prochoice", "fetus"
]

plk = [
    "abortion is murder", "unborn child", "killing babies", 'sanctity of life', "im prolife",
    "sanctity of life", "right to life", "protect the unborn", 
    "anti life", 'innocent', 'life begins at conception',
    "life liberty and the pursuit", "killing"
]


df_unlabeled['keyword_label'] = ""

def sent_score(text, pck, plk):
    pc_score = sum(1 for word in pck if word in text.lower())
    pl_score = sum(1 for word in plk if word in text.lower())

    if pc_score == 0 and pl_score == 0:
        return "no stance"
    if pc_score > pl_score:
        return "prochoice"
    if pc_score == pl_score:
        return "unclear"
    
    return "prolife"
    

# df_unlabeled['keyword_label'] = df_unlabeled['body'].apply(lambda x: sent_score(x, pck, plk))

/var/folders/c3/rmjxq8dj0rj3m6vr2txgfzfc0000gn/T/ipykernel_12976/4268730690.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_unlabeled['keyword_label'] = ""


In [23]:
pl_label = df_unlabeled[df_unlabeled['keyword_label'] == "prochoice"]
sample = pl_label.sample(150, random_state=13)

sample.to_csv('review.csv', index=False)

# checking counts

In [91]:
labeled_data['keyword_label'].value_counts()

keyword_label
prochoice    400
prolife      250
no stance    150
Name: count, dtype: int64

In [75]:
labeled_data['my_label'].unique()

correction = {
    'no since' : 'no stance',
    ' no stance': 'prolife',
    'unlcear': 'unclear'
}

labeled_data['my_label'] = labeled_data['my_label'].replace(correction)
labeled_data['my_label'] = labeled_data['my_label'].str.strip().str.lower()

In [93]:
labeled_data['my_label'].value_counts()

my_label
prolife      293
prochoice    260
no stance    190
unclear       57
Name: count, dtype: int64

In [92]:
labeled_data['my_label'] = labeled_data['my_label'].fillna(labeled_data['keyword_label'])

In [94]:
labeled_data.to_csv('labeled_data.csv', index=False)

**adding more politics**

In [57]:
pol = pd.read_parquet('politics_commentsClean.parquet')

In [58]:
pol = pol[~pol['body'].isin(['removed', 'deleted'])]

In [59]:
pol

,body,author,score,id,link_id,subreddit,author_flair_text,time
0,so if you find at 22 weeks that your fetus bra...,tau-lepton,12,ctexk5l,t3_3efts7,politics,None,2015-07-24 19:35:29
1,oh i get it. you dont actually know the quote....,Collypso,4,idont5t,t3_vk3jyt,politics,None,2022-06-25 13:23:03
2,the triumph of pointless interaction w faceles...,FoxGroundbreaking212,0,jk2pteh,t3_13gt8kz,politics,None,NaT
3,i think the fact that the fetus was created wi...,[deleted],1,c3hjhkl,t3_oihe2,politics,None,2012-01-15 23:33:46
4,shes been promised a ladyship in king trumps n...,[deleted],1,eq4vmcs,t3_bx78xg,politics,None,2019-06-06 02:10:08
...,...,...,...,...,...,...,...,...
24343,not really. they tended to take things a bit t...,Karanod,3,eqyraxp,t3_bzvdcc,politics,None,2019-06-13 01:08:33
24344,because thats how reddit rolls,seriously_icky,-4,g8ozgpo,t3_jadygn,politics,None,2020-10-13 14:12:16
24345,make all women of child bearing age register w...,Time_Theory_297,1,hwfvn7c,t3_sp45jk,politics,None,2022-02-11 00:30:54
24346,jesus that communist hippie talking about love...,Pilo5000,7,ietfawa,t3_vr5qq6,politics,None,2022-07-04 13:42:12


In [66]:
pol['keyword_label'] = pol['body'].apply(lambda x: sent_score(x, pck, plk))

In [81]:
review = pol[pol['keyword_label'] == 'no stance'].sample(150)

pol['keyword_label'].value_counts()

keyword_label
no stance    21061
prochoice      291
prolife        261
unclear         18
Name: count, dtype: int64

In [82]:
review.to_csv('review.csv', index=False)